# Fine-tune ViT5 trên Kaggle — tầng 3

Notebook này chạy `src/models/vit5.py` của repo DL-SummariseVN trên GPU T4 của Kaggle.

## Trước khi chạy: ba cài đặt ở menu **Settings** trên thanh công cụ

| Mục | Đặt thành | Vì sao |
|---|---|---|
| **Accelerator** | `GPU T4 x2` | Kaggle không có tuỳ chọn một T4; ta nhận hai rồi tự ghim còn một ở dưới |
| **Internet** | `On` | Cần tải `VietAI/vit5-base` và bộ `nam194/vietnews` từ Hugging Face. Muốn bật thì tài khoản Kaggle phải xác minh số điện thoại |
| **Persistence** | `Files only` (tuỳ chọn) | Giữ `/kaggle/working` giữa các phiên tương tác |

Quên một trong hai mục đầu thì ô kiểm tra môi trường sẽ dừng notebook ngay, chứ không
để nó huấn luyện trên CPU hàng giờ hay treo ở bước tải mô hình.

## Ngân sách

- **30 giờ GPU mỗi tuần**, đặt lại vào thứ Bảy — nhiều hơn Colab free đáng kể.
- **Một phiên tối đa 12 giờ**. `train_5k` 3 epoch hết khoảng 60 phút, `train_10k`
  khoảng 2 giờ, `train_20k` khoảng 4 giờ — đều gọn trong một phiên.
- Phiên tương tác tự ngắt khi để yên quá lâu. Cách chạy ngầm nằm ở cuối notebook.

## Cách dùng

Chỉ sửa **ô cấu hình** ngay dưới, rồi chạy toàn bộ. Mỗi version của notebook chạy
**một** cấu hình; muốn chạy cấu hình khác thì sửa ô đó và tạo version mới.

In [ ]:
# ==== CHI SUA O NAY ===================================================
# Moi version (Save & Run All) chay MOT cau hinh. Thu tu cho tuan 5:
#   1. train_5k   ~60 phut -- chay lai lan tuan 4; lan Colab truoc khong con
#                             file diem tung bai, nen chua so cap duoc voi ai
#   2. train_10k  ~2 gio
#   3. train_20k  ~4 gio
#   4. train_2k   ~25 phut -- diem dau duong cong hoc, neu con quota
# Khong gop nhieu cau hinh vao mot version: ba lan chay ~7,5 gio la sat tran
# 12 gio, va checkpoint cua ca ba (~6 GB moi lan) vuot gioi han output 20 GB.
MODEL = "VietAI/vit5-base"     # doi chung tuan 5: "vinai/bartpho-syllable"
TRAIN_SPLIT = "train_5k"
EPOCHS = 3
LR = 3e-5

DRY_RUN = True    # chay thu 5 buoc truoc: loi thu vien lo ra o phut 3, khong phai phut 60
RESUME = False    # True khi chay tiep mot lan bi cat giua chung -- xem muc cuoi notebook
# ======================================================================

REPO = "https://github.com/ICY825/SummariseVietNamese.git"
DIR = "/kaggle/working/BTL_DL"     # Kaggle chi cho ghi vao /kaggle/working
OUT = "/kaggle/working/runs"
RUN_DIR = f"{OUT}/{MODEL.replace('/', '_')}_{TRAIN_SPLIT}"   # dung cach vit5.py dat ten


def check(code, what):
    """Dung notebook neu lenh `!` ngay truoc do loi.

    `!lenh` loi KHONG nem ngoai le: notebook chay tiep cac o sau, va ban Save & Run
    All van bao thanh cong -- chi la khong co ket qua nao. IPython ghi ma thoat vao
    `_exit_code`; nem loi o day de Kaggle dung ngay tai cho hong.
    """
    if code != 0:
        raise RuntimeError(f"{what} THAT BAI (ma thoat {code}) -- xem log ngay tren.")
    print(f"{what}: OK")


print(f"Cau hinh: {MODEL} | {TRAIN_SPLIT} | {EPOCHS} epoch | lr {LR}"
      f" | chay thu {DRY_RUN} | resume {RESUME}")

In [ ]:
# Kiem tra moi truong. Hai loi hay gap nhat deu la loi cai dat chu khong phai loi code.
!nvidia-smi

import urllib.request
import datasets, torch, transformers
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("GPU thay duoc:", torch.cuda.device_count())

if torch.cuda.device_count() == 0:
    raise RuntimeError("Khong thay GPU. Vao Settings > Accelerator, chon GPU T4 x2.")
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception as e:
    raise RuntimeError(
        "Khong ra duoc Internet. Vao Settings > Internet, bat On "
        "(tai khoan Kaggle phai xac minh so dien thoai)."
    ) from e
print("Moi truong: OK")

## Vì sao phải ghim lại còn MỘT GPU

Kaggle cấp hai T4. `Trainer` thấy hai thiết bị sẽ tự bật DataParallel, và khi đó
`--batch 2` là **mỗi GPU** — batch hiệu dụng thành 32 chứ không phải 16 như README đã
chốt. Số liệu sinh ra sẽ không so được với lần chạy `train_5k` trên Colab, mà mục đích
của cả đề tài là **so sánh có kiểm soát**: đổi batch hiệu dụng giữa hai lần chạy là
đổi mất chính thứ đang được so.

Vì vậy mọi lệnh huấn luyện dưới đây đều mở đầu bằng `CUDA_VISIBLE_DEVICES=0`. Ghim
ngay trên dòng lệnh chứ không đặt biến môi trường của notebook, để chạy lại từng ô
theo thứ tự nào cũng đúng.

Muốn dùng cả hai T4 thì phải hạ `--grad-accum` xuống 4 để giữ batch hiệu dụng 16, và
ghi rõ điều đó trong báo cáo.

In [ ]:
import os
if not os.path.isdir(DIR):
    !git clone -q {REPO} {DIR}
    check(_exit_code, "git clone")

# `os.chdir` chu khong phai `%cd`: cac o `!` phia sau chay bang cwd cua kernel, va
# `os.chdir` doi dung cai do. `%cd {DIR}` phu thuoc vao viec magic co no chuoi bien
# hay khong, khac nhau giua cac ban IPython.
os.chdir(DIR)
!git pull -q
check(_exit_code, "git pull")
# Ma commit nay la thu duy nhat noi ket qua sinh tu phien ban code nao -- no nam
# trong log cua version, giu lai khi viet bao cao.
!git log --oneline -1
print("cwd:", os.getcwd())

Nếu repo để **private**, `git clone` sẽ hỏi mật khẩu rồi treo. Khi ấy vào
*Add-ons → Secrets*, lưu một GitHub token tên `GH_TOKEN`, rồi clone bằng:

```python
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("GH_TOKEN")
!git clone -q https://{tok}@github.com/ICY825/SummariseVietNamese.git {DIR}
```

In [ ]:
# Tu kiem tra: khong can mang, khong can GPU, xong trong vai giay.
# Chay truoc de biet chac ROUGE va bootstrap van dung trong moi truong Kaggle.
# `pipefail`: khong co no thi ma thoat la cua `tail` -- luon bang 0 -- va selftest
# hong van lot qua.
!bash -c "set -o pipefail; python src/eval/selftest.py | tail -3"
check(_exit_code, "selftest danh gia")
!bash -c "set -o pipefail; python src/models/selftest.py | tail -3"
check(_exit_code, "selftest tang 0-1")

## Chạy thử đường ống trước — 3 phút

`--max-steps 5` dừng huấn luyện sau 5 bước, `--eval-limit 20` chỉ sinh 20 bài. Không
lấy số liệu từ lần chạy này (tên file tự mang hậu tố `_thu20` để khỏi lẫn), mục đích
là để lỗi phiên bản thư viện hay lỗi tải dữ liệu lộ ra **bây giờ**, chứ không phải ở
phút thứ 60 của lần chạy thật.

Checkpoint của lần chạy thử ghi vào thư mục riêng rồi xoá ngay, để không lẫn với lần
chạy thật và không ăn vào giới hạn 20 GB của output.

In [ ]:
import shutil

if DRY_RUN and not RESUME:
    cmd = (f"CUDA_VISIBLE_DEVICES=0 python src/models/vit5.py --model {MODEL} "
           f"--train-split train_2k --eval-split val "
           f"--max-steps 5 --eval-limit 20 --out /kaggle/working/runs_thu")
    print(cmd)
    !{cmd}
    check(_exit_code, "chay thu duong ong")
    shutil.rmtree("/kaggle/working/runs_thu", ignore_errors=True)
else:
    print("Bo qua chay thu.")

## Chạy thật

`--out` phải trỏ vào `/kaggle/working`: đó là thư mục duy nhất được ghi và là thứ duy
nhất còn lại sau khi phiên kết thúc. Kết quả ra ba file — bảng chỉ số, bản dự đoán, và
`run.json` (hồ sơ lần chạy) — kèm một bản sao trong `--out` để sống sót khi ngắt phiên.

`fp16` tự bật vì T4 không hỗ trợ bf16. Nếu `eval_loss` thành `NaN` thì chạy lại với
`--fp16 off`, chậm hơn khoảng 33%.

Batch hiệu dụng 16 (`--batch 2 --grad-accum 8`) cố ý **không** nằm trong ô cấu hình:
đó là thứ phải giữ nguyên giữa mọi lần chạy của đường cong học.

In [ ]:
# Chi chay khi RESUME = True: chep checkpoint cua version truoc vao dung cho
# `vit5.py --resume` se tim. Truoc do phai them output cua version cu lam input:
# Add-ons > Add data > Your Work > chon notebook nay > version bi cat.
import glob

if RESUME:
    name = os.path.basename(RUN_DIR)
    found = sorted(glob.glob(f"/kaggle/input/**/{name}/checkpoint-*", recursive=True))
    os.makedirs(RUN_DIR, exist_ok=True)
    for ck in found:
        dest = os.path.join(RUN_DIR, os.path.basename(ck))
        if not os.path.exists(dest):
            print("chep", ck, "->", dest)
            shutil.copytree(ck, dest)
    if not glob.glob(f"{RUN_DIR}/checkpoint-*"):
        raise RuntimeError(
            f"RESUME = True nhung khong co checkpoint nao cua {name}, ca trong "
            "/kaggle/input lan /kaggle/working. Them output cua version cu lam input truoc."
        )
    print("San sang chay tiep tu:", sorted(glob.glob(f"{RUN_DIR}/checkpoint-*")))
else:
    print("Chay moi tu dau.")

In [ ]:
cmd = (f"CUDA_VISIBLE_DEVICES=0 python src/models/vit5.py --model {MODEL} "
       f"--train-split {TRAIN_SPLIT} --eval-split val "
       f"--epochs {EPOCHS} --lr {LR} --batch 2 --grad-accum 8 --out {OUT}"
       + (" --resume" if RESUME else ""))
print(cmd)
!{cmd}
check(_exit_code, f"huan luyen {TRAIN_SPLIT}")

In [ ]:
# Xem lai ket qua: bang chi so va ho so lan chay (bo qua cac lan chay thu `_thu`).
import json, pathlib

tables = sorted(p for p in pathlib.Path("results/tables").glob("*_run.json") if "_thu" not in p.name)
for p in tables:
    r = json.loads(p.read_text(encoding="utf-8"))
    print("=" * 70)
    print(p.name)
    print("  GPU        ", r["env"]["gpu"], "| fp16", r["env"]["fp16"])
    print("  train      ", r["data"]["train_split"], r["data"]["n_train"], "bai")
    print("  lr/epochs  ", r["args"]["lr"], "/", r["args"]["epochs"])
    print("  sinh       ", r["generation"])
    if r["train"]:
        print("  phut       ", r["train"]["minutes"])
        print("  checkpoint ", r["train"]["best_checkpoint"], "eval_loss", r["train"]["best_eval_loss"])
        ev = [(h.get("epoch"), h["eval_loss"]) for h in r["train"]["log_history"] if "eval_loss" in h]
        print("  eval_loss  ", ev)
    print("  rouge1     ", round(r["scores"]["corpus"]["rouge1"]["mean"], 2))
    print("  do dai     ", round(r["scores"]["length"]["mean_syllables"], 1), "am tiet")
    print("  so voi Lead-3:")
    print(r["versus_lead3"])

In [ ]:
# Gom ket qua cua lan chay nay (KHONG gom checkpoint) thanh mot file zip nho, de
# tai ve bang mot cu nhap o tab Output. Duong dan trong zip tinh tu goc repo, nen
# giai nen ngay tai thu muc BTL_DL tren may la file roi dung vao results/tables va
# results/predictions.
import zipfile

short = MODEL.split("/")[-1]
picked = sorted(
    p for p in pathlib.Path("results").rglob("*.json")
    if p.name.startswith(f"{short}-{TRAIN_SPLIT}_") and "_thu" not in p.name
)
if not picked:
    raise RuntimeError(f"Khong thay file ket qua nao cua {short}-{TRAIN_SPLIT} trong results/.")
zpath = f"/kaggle/working/ket_qua_{short}_{TRAIN_SPLIT}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in picked:
        z.write(p, p.as_posix())
        print(f"  {p.as_posix():90s} {p.stat().st_size / 1e6:6.2f} MB")
print("Da dong goi:", zpath)

## Train ngầm — đóng trình duyệt vẫn chạy tiếp

Phiên tương tác chết khi mất mạng hoặc khi để yên quá lâu. Kaggle có sẵn cách chạy
tách rời, và đây là điểm hơn hẳn Colab free:

**Save Version → chọn `Save & Run All (Commit)` → Save.**

Kaggle chép notebook sang một máy khác rồi chạy **toàn bộ các ô từ đầu đến cuối**,
không cần trình duyệt mở. Tắt máy đi ngủ cũng được. Vài điều phải nhớ:

- Bản chạy ngầm bắt đầu từ môi trường sạch, nên ô `git clone` và ô tải dữ liệu đều
  chạy lại. Không ô nào chờ nhập tay.
- Ô nào hỏng thì version dừng ngay tại đó và báo **lỗi** — không có chuyện chạy hết
  mà không có kết quả.
- Xong thì vào tab **Output** của version đó, tải `ket_qua_<mô hình>_<tập>.zip` về rồi
  giải nén tại thư mục gốc của repo. Checkpoint nằm trong `runs/`, chỉ tải khi cần.
- Giới hạn vẫn là **12 giờ** và output **20 GB**.
- Theo dõi tiến độ ở trang notebook, mục *Versions* — log in ra được xem trực tiếp.

Muốn chắc chắn không mất công, trước khi Commit hãy chạy tay đến hết ô "chạy thử
đường ống" một lần: một bản Commit hỏng ở phút thứ 50 vẫn tiêu tốn đúng ngần ấy giờ
trong quota.

In [ ]:
# TUY CHON — chi chay khi da tai ket qua ve, hoac khi output qua nang.
# Moi `checkpoint-*` cua ViT5-base nang khoang 2,7 GB (trong so + trang thai optimizer);
# `save_total_limit=2` nen co the ton ~5,4 GB, du sat gioi han 20 GB cua output.
# `final/` la ban da duoc `load_best_model_at_end` chon, giu lai la du de tuan 5 khao
# sat tham so sinh bang `--no-train --model <duong dan final>`.
!du -sh /kaggle/working/runs/* 2>/dev/null
# !rm -rf /kaggle/working/runs/*/checkpoint-*

## Chạy tiếp khi bị cắt giữa chừng

`vit5.py` ghi checkpoint sau mỗi epoch, nên lần chạy sau nối tiếp được. Trên Kaggle,
checkpoint của phiên trước không tự có mặt:

1. *Add-ons → Add data → Your Work*, chọn notebook này và **version bị cắt** để thêm
   output của nó làm input (nó nằm ở `/kaggle/input/...`).
2. Trong ô cấu hình đặt `RESUME = True`, giữ nguyên `MODEL` và `TRAIN_SPLIT`.
3. Chạy lại toàn bộ. Ô chuẩn bị tự tìm và chép checkpoint vào đúng chỗ, và báo lỗi
   nếu không tìm thấy thay vì lặng lẽ huấn luyện lại từ đầu.

Với `train_5k` (60 phút) thì việc này hầu như không cần; nó dành cho `train_20k`.